## Data Preparation and Cleaning

In [1]:
import numpy as np

import pandas as pd
import yfinance as yf

from lets_plot import *
LetsPlot.setup_html()

In [2]:
data_precovid =\
    (
        yf.download(["NVDA","AMD"], start="2015-01-01", end="2020-02-19", auto_adjust=True)["Close"]
    )
data_precovid_rebased =\
(
    data_precovid
    .assign(nvda_rebased=lambda d: 100 * d["NVDA"] / d["NVDA"].iloc[0],
            amd_rebased=lambda d: 100 * d["AMD"] / d["AMD"].iloc[0])
    .drop(columns = ["NVDA","AMD"])
            
)

[*********************100%***********************]  2 of 2 completed


In [3]:
data_precovid_long =\
(
    data_precovid
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)
data_precovid_long_rebased =\
(
    data_precovid_rebased
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)

(
    ggplot(data_precovid_long, aes(x = "date", y = "price"))
    +geom_line(aes(color = "ticker"))
    +ggsize(1000,500)
    +ggtitle("NVDA vs AMD, pre-COVID, original")
)

In [4]:
(
    ggplot(data_precovid_long_rebased, aes(x="date", y="price"))
    + geom_line(aes(color="ticker"))
    + ggsize(1000, 500)
    +ggtitle("NVDA vs AMD, pre-COVID, base 100")
)

In [5]:
data_covid =\
    (
        yf.download(["NVDA","AMD"], start="2020-02-20", end="2022-11-29", auto_adjust=True)["Close"]
    )
data_covid_rebased =\
(
    data_covid
    .assign(nvda_rebased=lambda d: 100 * d["NVDA"] / d["NVDA"].iloc[0],
            amd_rebased=lambda d: 100 * d["AMD"] / d["AMD"].iloc[0])
    .drop(columns = ["NVDA","AMD"])
            
)

[*********************100%***********************]  2 of 2 completed


In [6]:
data_covid_long =\
(
    data_covid
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)
data_covid_long_rebased =\
(
    data_covid_rebased
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)

(
    ggplot(data_covid_long, aes(x = "date", y = "price"))
    +geom_line(aes(color = "ticker"))
    +ggsize(1000,500)
    +ggtitle("NVDA vs AMD, COVID, original")
)

In [7]:
(
    ggplot(data_covid_long_rebased, aes(x="date", y="price"))
    + geom_line(aes(color="ticker"))
    + ggsize(1000, 500)
    +ggtitle("NVDA vs AMD, COVID, base 100")
)

In [8]:
data_aiboom =\
    (
        yf.download(["NVDA","AMD"], start="2022-11-30", end="2026-09-12", auto_adjust=True)["Close"]
    )
data_aiboom_rebased =\
(
    data_aiboom
    .assign(nvda_rebased=lambda d: 100 * d["NVDA"] / d["NVDA"].iloc[0],
            amd_rebased=lambda d: 100 * d["AMD"] / d["AMD"].iloc[0])
    .drop(columns = ["NVDA","AMD"])
            
)

[*********************100%***********************]  2 of 2 completed


In [9]:
data_aiboom_long =\
(
    data_aiboom
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)
data_aiboom_long_rebased =\
(
    data_aiboom_rebased
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "ticker", value_name= "price")
)

(
    ggplot(data_aiboom_long, aes(x = "date", y = "price"))
    +geom_line(aes(color = "ticker"))
    +ggsize(1000,500)
    +ggtitle("NVDA vs AMD, AI boom, original")
)

In [10]:
(
    ggplot(data_aiboom_long_rebased, aes(x="date", y="price"))
    + geom_line(aes(color="ticker"))
    + ggsize(1000, 500)
    +ggtitle("NVDA vs AMD, AI boom, base 100")
)

## Prepare one continuous price series

In [24]:
# Additional imports for pair validation
# The original imports, including Lets-Plot, remain unchanged.
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller

prices = (
    pd.concat([
        data_precovid,
        data_covid,
        data_aiboom
    ])
    .sort_index()
    .loc[lambda data: ~data.index.duplicated(keep="first")]
    .dropna()
)

log_prices = np.log(prices)

print(f"Common trading days: {len(prices):,}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")

display(prices.head())
display(prices.tail())

Common trading days: 2,938
Date range: 2015-01-02 to 2026-09-11


Ticker,AMD,NVDA
Date,,
2015-01-02,2.67,0.481884
2015-01-05,2.66,0.473745
2015-01-06,2.63,0.459382
2015-01-07,2.58,0.458184
2015-01-08,2.61,0.475420


Ticker,AMD,NVDA
Date,,
2026-09-04,477.570007,230.102524
2026-09-08,505.739990,225.477692
2026-09-09,521.099976,223.419998
2026-09-10,503.600006,218.360001
2026-09-11,516.130005,218.289993


## Step 1 — Cointegration and spread-stability diagnostics


In [17]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller

In [25]:
daily_returns = prices.pct_change().dropna()
return_correlation = daily_returns["AMD"].corr(daily_returns["NVDA"])

# Engle–Granger test:
# Null hypothesis = AMD and NVIDIA are not cointegrated.
coint_stat, coint_pvalue, _ = coint(
    log_prices["AMD"],
    log_prices["NVDA"]
)

# Static log-price regression for a diagnostic spread
static_model = sm.OLS(
    log_prices["AMD"],
    sm.add_constant(log_prices["NVDA"])
).fit()

static_alpha = static_model.params["const"]
static_beta = static_model.params["NVDA"]

static_spread = static_model.resid.rename("static_log_spread")

# ADF test:
# Null hypothesis = the residual spread is non-stationary.
adf_stat, adf_pvalue, *_ = adfuller(
    static_spread.dropna(),
    autolag="AIC"
)

diagnostics = pd.Series({
    "daily return correlation": return_correlation,
    "Engle–Granger statistic": coint_stat,
    "Engle–Granger p-value": coint_pvalue,
    "static hedge intercept (alpha)": static_alpha,
    "static hedge ratio (beta)": static_beta,
    "ADF statistic for residual spread": adf_stat,
    "ADF p-value for residual spread": adf_pvalue,
})

display(diagnostics.to_frame("value"))

if coint_pvalue < 0.05 and adf_pvalue < 0.05:
    print(
        "Diagnostic result: the full-sample spread shows evidence "
        "consistent with stationarity."
    )
else:
    print(
        "Diagnostic result: full-sample evidence of a stationary spread "
        "is weak; interpret any later strategy result cautiously."
    )

,value
daily return correlation,0.569988
Engle–Granger statistic,-1.602595
Engle–Granger p-value,0.719793
static hedge intercept (alpha),1.731948
static hedge ratio (beta),0.782862
ADF statistic for residual spread,-1.602264
ADF p-value for residual spread,0.482456


Diagnostic result: full-sample evidence of a stationary spread is weak; interpret any later strategy result cautiously.


## Step 2 — Rolling hedge ratio

**Idea：** The hedge ratio tells us how much NVIDIA exposure is needed to offset AMD exposure in the estimated relationship. If beta is 0.8, the diagnostic spread compares AMD with approximately 0.8 units of NVIDIA in log-price terms.

**Why：** The relationship between two technology stocks can change across market regimes. A rolling estimate adapts over time instead of assuming that one full-sample coefficient is always correct.

**Anti-look-ahead rule：** At each date *t*, the regression uses only the previous 60 trading days (`t-60` to `t-1`). The resulting log spread is `log(AMD) − alpha_t − beta_t × log(NVDA)`.

In [26]:
HEDGE_LOOKBACK = 60

rolling_alpha = pd.Series(
    index=log_prices.index,
    dtype=float,
    name="rolling_alpha"
)

rolling_beta = pd.Series(
    index=log_prices.index,
    dtype=float,
    name="rolling_hedge_ratio"
)

rolling_spread = pd.Series(
    index=log_prices.index,
    dtype=float,
    name="rolling_log_spread"
)

for i in range(HEDGE_LOOKBACK, len(log_prices)):
    # Only use information available before date t
    estimation_window = log_prices.iloc[i - HEDGE_LOOKBACK:i]

    model = sm.OLS(
        estimation_window["AMD"],
        sm.add_constant(estimation_window["NVDA"])
    ).fit()

    alpha_t = model.params["const"]
    beta_t = model.params["NVDA"]

    rolling_alpha.iloc[i] = alpha_t
    rolling_beta.iloc[i] = beta_t

    rolling_spread.iloc[i] = (
        log_prices["AMD"].iloc[i]
        - alpha_t
        - beta_t * log_prices["NVDA"].iloc[i]
    )

display(
    pd.concat(
        [rolling_alpha, rolling_beta, rolling_spread],
        axis=1
    ).tail()
)

,rolling_alpha,rolling_hedge_ratio,rolling_log_spread
Date,,,
2026-09-04,8.479250,-0.423409,-0.007817
2026-09-08,8.704498,-0.465214,0.042159
2026-09-09,8.626129,-0.450302,0.065520
2026-09-10,8.475849,-0.422013,0.018955
2026-09-11,8.483251,-0.423619,0.044644


In [27]:
hedge_plot_data = (
    rolling_beta
    .reset_index()
    .rename(columns={
        "Date": "date",
        "index": "date",
        "rolling_hedge_ratio": "hedge_ratio"
    })
    .dropna()
)

ggplot(hedge_plot_data, aes(x="date", y="hedge_ratio")) + \
    geom_line(color="#1f77b4") + \
    geom_hline(
        yintercept=static_beta,
        color="black",
        linetype="dashed"
    ) + \
    labs(
        title="Rolling 60-Day Hedge Ratio: AMD versus NVIDIA",
        x="Date",
        y="Hedge ratio"
    ) + \
    ggsize(900, 350)

## Step 3 — Rolling Z-Score

**Idea：** A spread alone is difficult to interpret because its scale can change. The z-score standardises the spread relative to its recent history.

**Why：** It provides a comparable signal for later entry and exit rules. A z-score of +2 means that the spread is two recent-window standard deviations above its recent average.

**Timing：** The z-score uses only the previous 60 spreads. If the group later creates positions, today’s z-score must affect a trade no earlier than the next trading day.

In [28]:
ZSCORE_LOOKBACK = 60

rolling_zscore = pd.Series(
    index=log_prices.index,
    dtype=float,
    name="rolling_zscore"
)

for i in range(HEDGE_LOOKBACK + ZSCORE_LOOKBACK, len(log_prices)):
    # Use only prior spreads, not today's spread, for calibration
    prior_spreads = rolling_spread.iloc[
        i - ZSCORE_LOOKBACK:i
    ].dropna()

    prior_std = prior_spreads.std(ddof=0)

    if len(prior_spreads) == ZSCORE_LOOKBACK and prior_std > 0:
        rolling_zscore.iloc[i] = (
            rolling_spread.iloc[i] - prior_spreads.mean()
        ) / prior_std

features = pd.concat(
    [
        prices,
        rolling_alpha,
        rolling_beta,
        rolling_spread,
        rolling_zscore
    ],
    axis=1
)

display(features.tail(10))

,AMD,NVDA,rolling_alpha,rolling_hedge_ratio,rolling_log_spread,rolling_zscore
Date,,,,,,
2026-08-28,465.579987,217.306839,7.362543,-0.213276,-0.071552,-1.215980
2026-08-31,470.720001,220.533234,7.607396,-0.259625,-0.052182,-1.071096
2026-09-01,459.609985,217.196976,7.852474,-0.305869,-0.076274,-1.168282
2026-09-02,457.059998,224.159180,8.048108,-0.342509,-0.069513,-1.093619
2026-09-03,456.160004,228.194656,8.258627,-0.382080,-0.061012,-1.013312
2026-09-04,477.570007,230.102524,8.479250,-0.423409,-0.007817,-0.703671
2026-09-08,505.739990,225.477692,8.704498,-0.465214,0.042159,-0.416211
2026-09-09,521.099976,223.419998,8.626129,-0.450302,0.065520,-0.272097
2026-09-10,503.600006,218.360001,8.475849,-0.422013,0.018955,-0.499502


In [29]:
spread_plot_data = (
    rolling_spread
    .reset_index()
    .rename(columns={
        "Date": "date",
        "index": "date",
        "rolling_log_spread": "spread"
    })
    .dropna()
)

zscore_plot_data = (
    rolling_zscore
    .reset_index()
    .rename(columns={
        "Date": "date",
        "index": "date",
        "rolling_zscore": "zscore"
    })
    .dropna()
)

spread_chart = (
    ggplot(spread_plot_data, aes(x="date", y="spread"))
    + geom_line(color="#1f77b4")
    + labs(
        title="Rolling Log Spread",
        x="Date",
        y="Log spread"
    )
    + ggsize(900, 300)
)

zscore_chart = (
    ggplot(zscore_plot_data, aes(x="date", y="zscore"))
    + geom_line(color="#6a3d9a")
    + geom_hline(yintercept=0, color="black")
    + geom_hline(yintercept=2, color="#d62728", linetype="dashed")
    + geom_hline(yintercept=-2, color="#2ca02c", linetype="dashed")
    + labs(
        title="Rolling 60-Day Z-Score",
        x="Date",
        y="Z-score"
    )
    + ggsize(900, 350)
)

spread_chart
zscore_chart